In [0]:
# ==========================================
# 04 - GOLD AGGREGATION
# Fraud Detection Lakehouse
# ==========================================

from pyspark.sql import functions as F

CATALOG_NAME = "fraud_detection"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG_NAME}")

print("Gold aggregation initialized")

In [0]:
# Load validated Silver tables

customers = spark.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers"
)

merchants = spark.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.merchants"
)

transactions = spark.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions"
)

print("Customers:", customers.count())
print("Merchants:", merchants.count())
print("Transactions:", transactions.count())

In [0]:
# ==========================================
# CUSTOMER RISK METRICS
# ==========================================

customer_metrics = (
    transactions
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("total_transactions"),

        F.sum("amount").alias("total_transaction_amount"),

        F.avg("amount").alias("average_transaction_amount"),

        F.sum("fraud_label").alias("fraudulent_transactions"),

        F.sum(
            F.when(F.col("amount") >= 50000, 1).otherwise(0)
        ).alias("high_value_transactions"),

        F.countDistinct("merchant_id").alias("unique_merchants"),

        F.min("transaction_timestamp").alias("first_transaction"),

        F.max("transaction_timestamp").alias("last_transaction")
    )
)

# Include customers with no transactions

customer_risk = (
    customers
    .join(customer_metrics, on="customer_id", how="left")
    .fillna({
        "total_transactions": 0,
        "total_transaction_amount": 0,
        "average_transaction_amount": 0,
        "fraudulent_transactions": 0,
        "high_value_transactions": 0,
        "unique_merchants": 0
    })
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("fraudulent_transactions") /
            F.col("total_transactions")
        ).otherwise(0)
    )
)

# Rule-based risk categorization

customer_risk = customer_risk.withColumn(
    "risk_category",
    F.when(
        (F.col("fraudulent_transactions") >= 2) |
        (
            (F.col("fraudulent_transactions") >= 1) &
            (F.col("fraud_rate") >= 0.10)
        ),
        "HIGH"
    ).when(
        (F.col("fraudulent_transactions") == 1) |
        (F.col("high_value_transactions") >= 3),
        "MEDIUM"
    ).otherwise("LOW")
)

customer_risk = customer_risk.withColumn(
    "risk_assessment_timestamp",
    F.current_timestamp()
)

# Save as Gold Delta table

(
    customer_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG_NAME}.{GOLD_SCHEMA}.customer_risk_metrics"
    )
)

print("Customer risk metrics generated successfully!")

In [0]:
customer_risk_gold = spark.table(
    "fraud_detection.gold.customer_risk_metrics"
)

print("Total customers:", customer_risk_gold.count())

display(
    customer_risk_gold.limit(20)
)

In [0]:
display(
    customer_risk_gold
    .groupBy("risk_category")
    .count()
    .orderBy("risk_category")
)